<a href="https://colab.research.google.com/github/SehrishbAsghar/FlyRank_ML_Internship_Sehrish/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

raw = con.sql(f"""
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
""").df()

content = con.sql(f"""
    SELECT content_hash_id, content_type
    FROM read_parquet('{rel}/dim_content.parquet')
""").df()

df = raw.merge(content, on="content_hash_id", how="left")
df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"].replace(0, np.nan)
df["ctr"] = df["ctr"].fillna(0)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [2]:
key_fields = ["gsc_impressions", "gsc_clicks", "gsc_avg_position", "ctr"]

print("=== Summary statistics ===")
print(df[key_fields].describe())

print("\n=== Percentiles (to catch heavy tails) ===")
for col in key_fields:
    p = df[col].quantile([0.5, 0.9, 0.95, 0.99, 0.999, 1.0])
    print(f"\n{col}:")
    print(p)

=== Summary statistics ===
       gsc_impressions    gsc_clicks  gsc_avg_position           ctr
count     3.611061e+06  3.611061e+06      3.611061e+06  3.611061e+06
mean      7.772164e+01  2.275874e-01      1.582665e+01  3.080748e-03
std       2.498747e+02  1.277267e+00      1.985603e+01  3.009151e-02
min       1.000000e+00  0.000000e+00      0.000000e+00  0.000000e+00
25%       4.000000e+00  0.000000e+00      3.742120e+00  0.000000e+00
50%       1.600000e+01  0.000000e+00      7.500000e+00  0.000000e+00
75%       6.200000e+01  0.000000e+00      2.020000e+01  0.000000e+00
max       4.008400e+04  2.740000e+02      4.980000e+02  1.000000e+00

=== Percentiles (to catch heavy tails) ===

gsc_impressions:
0.500       16.0
0.900      185.0
0.950      335.0
0.990      942.0
0.999     2700.0
1.000    40084.0
Name: gsc_impressions, dtype: float64

gsc_clicks:
0.500      0.0
0.900      1.0
0.950      1.0
0.990      4.0
0.999     12.0
1.000    274.0
Name: gsc_clicks, dtype: float64

gsc_avg_posit

## 1. Distributions

**gsc_impressions:**

median 16, mean 77.7 (inflated by outliers), max 40,084.
Extremely heavy right tail even the 99.9th percentile (2,700) is a tiny fraction of the max. A small number of pages dominate total impression volume.

**gsc_clicks:**

median 0 the majority of page-days get zero clicks at all.
90th percentile is still just 1 click; max is 274, an extreme outlier.

**gsc_avg_position:**

median 7.5 (front-page territory), but a long tail out to position 498. The 99th percentile (88.75) already reflects deeply buried results.

**ctr:**

Median 0, since most rows have 0 clicks. Max is exactly 1.0 (100% CTR) almost certainly driven by very low-impression rows (e.g. 1 impression, 1 click), not genuine strong performance.

**Takeaway:**

Every key field here has a heavy right tail, and zero/near-zero is
the "typical" case, not the exception, for clicks and CTR. Any signal audit or rule built on averages needs to account for this, a plain mean will be dominated by a small number of extreme rows, and CTR values near 1.0 need sample-size context
before being trusted.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [3]:
schema_check = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/dim_content.parquet') LIMIT 1").df()
print(schema_check)

                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         cpc      DOUBLE  YES 

In [4]:
content_full = con.sql(f"""
    SELECT content_hash_id, content_type, content_updated_date, last_optimized_date
    FROM read_parquet('{rel}/dim_content.parquet')
""").df()

df = raw.merge(content_full, on="content_hash_id", how="left")
df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"].replace(0, np.nan)
df["ctr"] = df["ctr"].fillna(0)

def position_tier(pos):
    if pd.isna(pos): return "unknown"
    elif pos <= 3: return "top_3"
    elif pos <= 10: return "page_1"
    elif pos <= 20: return "striking"
    elif pos <= 50: return "page_3_5"
    else: return "deep"

df["position_tier"] = df["gsc_avg_position"].apply(position_tier)

# Staleness: days between report_date and content_updated_date
df["report_date"] = pd.to_datetime(df["report_date"])
df["content_updated_date"] = pd.to_datetime(df["content_updated_date"])
df["days_since_update"] = (df["report_date"] - df["content_updated_date"]).dt.days

In [5]:
# --- Signal test #1: CTR vs position_tier ---
signal1 = df.groupby("position_tier")["ctr"].agg(["mean", "count"]).round(4)
print("Signal 1 — CTR vs position_tier:")
print(signal1)

# --- Signal test #2: CTR vs impression volume ---
df["volume_bucket"] = pd.qcut(df["gsc_impressions"], q=4, labels=["low", "mid_low", "mid_high", "high"], duplicates="drop")
signal2 = df.groupby("volume_bucket", observed=True)["ctr"].agg(["mean", "count"]).round(4)
print("\nSignal 2 — CTR vs volume bucket:")
print(signal2)

# --- Signal test #3: CTR vs staleness (days since last update) ---
df["staleness_bucket"] = pd.cut(df["days_since_update"],
                                   bins=[-1, 30, 90, 180, 365, 100000],
                                   labels=["0-30d", "31-90d", "91-180d", "181-365d", "365d+"])
signal3 = df.groupby("staleness_bucket", observed=True)["ctr"].agg(["mean", "count"]).round(4)
print("\nSignal 3 — CTR vs staleness bucket:")
print(signal3)

Signal 1 — CTR vs position_tier:
                 mean    count
position_tier                 
deep           0.0005   276863
page_1         0.0035  1456122
page_3_5       0.0016   631491
striking       0.0028   519223
top_3          0.0048   727362

Signal 2 — CTR vs volume bucket:
                 mean   count
volume_bucket                
low            0.0041  973112
mid_low        0.0024  868861
mid_high       0.0027  873000
high           0.0031  896088

Signal 3 — CTR vs staleness bucket:
                    mean   count
staleness_bucket                
0-30d             0.0023  553364
31-90d            0.0019   84890
91-180d           0.0121    9087
181-365d          0.0024    1404


**Signal 1: CTR vs. position_tier** CONFIRMED. CTR declines monotonically and
cleanly from top_3 (0.0048) to deep (0.0005), across large samples (277K-1.46M rows per bucket). Solid, trustworthy signal.

**Signal 2: CTR vs. volume bucket** MIXED. No monotonic relationship (low:
0.0041, mid_low: 0.0024, mid_high: 0.0027, high: 0.0031) volume doesn't predict CTR level directly; it should only scale opportunity size, not signal direction.

**Signal 3: CTR vs. staleness (days since last update)** MIXED / mostly FALSE as a standalone signal, for two reasons. First, only 648,745 of 3,611,061 rows
(~18%) have a non-null `content_updated_date` most content in this slice has
no recorded update date at all, which alone should stop this signal from being used broadly without first handling that gap. Second, among the rows that do have a date, the pattern isn't monotonic: 0-30d (0.0023) then 31-90d (0.0019) then 91-180d
(0.0121, a spike) then 181-365d (0.0024). The 91-180d spike sits on a tiny sample (9,087 rows vs hundreds of thousands elsewhere) and is likely noise, not a real
effect.

A clearly-explained negative here is a real win: it means a refresh-flag rule based purely on "days since update" would be unreliable on this slice both
because of missing dates and because the CTR-vs-staleness relationship isn't clean
even where dates exist.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [6]:
# Control for position_tier while checking staleness -- does the effect hold within each tier?
flag_test = df[df["days_since_update"].notna()].groupby(
    ["position_tier", "staleness_bucket"], observed=True
)["ctr"].agg(["mean", "count"]).round(4)

print(flag_test)

                                  mean   count
position_tier staleness_bucket                
deep          0-30d             0.0004   30915
              31-90d            0.0006    4450
              91-180d           0.0002    1302
              181-365d          0.0000     171
page_1        0-30d             0.0026  205965
              31-90d            0.0021   32132
              91-180d           0.0088    3535
              181-365d          0.0021     685
page_3_5      0-30d             0.0012  111839
              31-90d            0.0011   16469
              91-180d           0.0012    1419
              181-365d          0.0000     132
striking      0-30d             0.0022   78795
              31-90d            0.0018   11690
              91-180d           0.0083    1038
              181-365d          0.0000     127
top_3         0-30d             0.0030  125850
              31-90d            0.0024   20149
              91-180d           0.0382    1793
             

**Flag tested:** FlyRank's refresh flag, which assumes older content (more days since last update) has lower CTR and should be prioritized for refresh.

**Test:** controlled for position_tier while checking CTR across staleness buckets, to see if the staleness effect holds independent of position (rather
than being confounded by it).

**Result:** the assumption does NOT hold cleanly. Within every position tier, 0-30d and 31-90d CTR are close together with no strong decline (e.g. page_1:
0.0026 to 0.0021; top_3: 0.0030 to 0.0024). The 91-180d bucket actually spikes upward in most tiers (top_3: 0.0382, its highest value in the entire table),
then drops again at 181-365d. This is the opposite of "staler = worse

**Verdict: OPPOSITE / FALSE as a standalone rule basis.** The 91-180d spike sits on small samples (1,038-3,535 rows vs tens of thousands in the 0-30d buckets), so it's likely noise rather than a genuine "content ages into a sweet spot" effect,
but either way, the data does not support "the older, the worse," which is the refresh flag's implicit assumption. Combined with Section 2's finding that ~82%
of rows have no update date at all, this is a real caveat worth surfacing: a refresh-priority rule built purely on days-since-update would be acting on a weak, possibly backwards signal for this slice of data.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

A content team relying on "days since last update" to prioritize refresh work would be acting on a weak or misleading signal for this slice. staleness alone doesn't predict CTR here, and the one bucket that looked promising (91-180 days) is backed by too little data to trust.

Refresh decisions should lean instead on the confirmed signal CTR relative to position tier and treat staleness as, at best, a secondary factor to investigate case-by-case, not a standalone trigger for which pages get refreshed first.

## Self-check



- Every section above is filled — markdown thinking AND the code that backs it
- The notebook runs top to bottom with no errors (Runtime → Run all)
- No client names, URLs, or private queries anywhere
- My claims use careful words: observed, measured, directional, decision-support
- Committed to my repo under `work/notebooks/`